In [1]:
# ==========================================================
# PROJECT PATH SETUP
# ==========================================================

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project Root:", PROJECT_ROOT)

Project Root: d:\Kavya Shankar_Workspace\Kaggle Projects\Predicting Smartphone Addiction


In [2]:
# ==========================================================
# STEP 01 | LOAD + FEATURE ENGINEERING
# ==========================================================

from src.data_loader import load_data
from src.feature_engineering import engineer_features

train_df, test_df, sample_submission = load_data()

train_fe = engineer_features(train_df)
test_fe = engineer_features(test_df)

print("Train Shape:", train_fe.shape)
print("Test Shape :", test_fe.shape)

Train Shape: (691369, 23)
Test Shape : (296302, 22)


In [3]:
# ==========================================================
# STEP 02 | SEPARATE FEATURES AND TARGET
# ==========================================================

test_ids = test_fe["id"].copy()

X = train_fe.drop(columns=["id", "addicted_label"])
y = train_fe["addicted_label"]

X_test = test_fe.drop(columns=["id"])

print("X      :", X.shape)
print("y      :", y.shape)
print("X_test :", X_test.shape)

X      : (691369, 21)
y      : (691369,)
X_test : (296302, 21)


In [4]:
# ==========================================================
# STEP 03 | STRATIFIED TRAIN / VALIDATION SPLIT
# ==========================================================

from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)

X_train: (553095, 21)
X_valid: (138274, 21)


In [5]:
# ==========================================================
# STEP 04 | BUILD PREPROCESSOR
# ==========================================================

from src.preprocessing import build_preprocessor

numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

preprocessor = build_preprocessor(
    numerical_features=numerical_features,
    categorical_features=categorical_features,
)

print("Numerical Features   :", len(numerical_features))
print("Categorical Features :", len(categorical_features))
print("Preprocessor created.")

Numerical Features   : 18
Categorical Features : 3
Preprocessor created.


In [6]:
# ==========================================================
# STEP 05 | PREPROCESS DATA
# ==========================================================

X_train_processed = preprocessor.fit_transform(X_train)

X_valid_processed = preprocessor.transform(X_valid)

X_test_processed = preprocessor.transform(X_test)

print("X_train processed:", X_train_processed.shape)
print("X_valid processed:", X_valid_processed.shape)
print("X_test processed :", X_test_processed.shape)

X_train processed: (553095, 26)
X_valid processed: (138274, 26)
X_test processed : (296302, 26)


In [7]:
# ==========================================================
# STEP 06 | BASELINE MODEL — LOGISTIC REGRESSION
# ==========================================================

from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(X_train_processed, y_train)

print("Logistic Regression training completed.")

Logistic Regression training completed.


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [8]:
# ==========================================================
# STEP 07 | VALIDATION PREDICTIONS
# ==========================================================

y_valid_proba = logistic_model.predict_proba(
    X_valid_processed
)[:, 1]

print("Validation predictions:", y_valid_proba.shape)
print("Minimum probability   :", y_valid_proba.min())
print("Maximum probability   :", y_valid_proba.max())

Validation predictions: (138274,)
Minimum probability   : 0.0003732853219785658
Maximum probability   : 0.9999474847115625


In [9]:
# ==========================================================
# STEP 08 | BASELINE ROC-AUC
# ==========================================================

from sklearn.metrics import roc_auc_score

baseline_auc = roc_auc_score(
    y_valid,
    y_valid_proba
)

print(f"Baseline ROC-AUC: {baseline_auc:.6f}")

Baseline ROC-AUC: 0.917445


In [10]:
# ==========================================================
# STEP 09 | LOGISTIC REGRESSION FEATURE IMPORTANCE
# ==========================================================

import pandas as pd
import numpy as np

processed_feature_names = preprocessor.get_feature_names_out()

coefficients = logistic_model.coef_[0]

feature_importance = pd.DataFrame({
    "feature": processed_feature_names,
    "coefficient": coefficients,
    "absolute_coefficient": np.abs(coefficients)
})

feature_importance = feature_importance.sort_values(
    "absolute_coefficient",
    ascending=False
)

feature_importance.head(15)

,feature,coefficient,absolute_coefficient
14,numerical__work_study_ratio,-5.926840,5.926840
13,numerical__gaming_ratio,-5.383891,5.383891
12,numerical__social_media_ratio,3.210501,3.210501
16,numerical__screen_time_waking_ratio,-1.891359,1.891359
25,categorical__academic_work_impact_Yes,-1.104811,1.104811
24,categorical__academic_work_impact_No,-1.082188,1.082188
2,numerical__social_media_hours,0.809838,0.809838
20,categorical__gender_Other,-0.756250,0.756250
21,categorical__stress_level_High,-0.747754,0.747754
22,categorical__stress_level_Low,-0.742344,0.742344


In [11]:
print("TOP POSITIVE FEATURES")
print(
    feature_importance
    .sort_values("coefficient", ascending=False)
    .head(10)[["feature", "coefficient"]]
)

print("\nTOP NEGATIVE FEATURES")
print(
    feature_importance
    .sort_values("coefficient")
    .head(10)[["feature", "coefficient"]]
)

TOP POSITIVE FEATURES
                               feature  coefficient
12       numerical__social_media_ratio     3.210501
2        numerical__social_media_hours     0.809838
3              numerical__gaming_hours     0.670676
11      numerical__entertainment_ratio     0.551429
4          numerical__work_study_hours     0.543946
1   numerical__daily_screen_time_hours     0.351288
8       numerical__weekend_screen_time     0.304423
10       numerical__weekend_difference     0.095633
5               numerical__sleep_hours     0.078119
15    numerical__leisure_to_work_ratio     0.005378

TOP NEGATIVE FEATURES
                                  feature  coefficient
14            numerical__work_study_ratio    -5.926840
13                numerical__gaming_ratio    -5.383891
16    numerical__screen_time_waking_ratio    -1.891359
25  categorical__academic_work_impact_Yes    -1.104811
24   categorical__academic_work_impact_No    -1.082188
20              categorical__gender_Other    -0.75625

In [12]:
# ==========================================================
# EXPERIMENT 02 | CHECK CATBOOST
# ==========================================================

try:
    import catboost
    print("CatBoost version:", catboost.__version__)
except ImportError:
    print("CatBoost is not installed.")

CatBoost version: 1.2.10


In [13]:
# ==========================================================
# EXPERIMENT 02 | CATBOOST
# STEP 01 | CATEGORICAL FEATURES
# ==========================================================

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Categorical Features:")
print(categorical_features)

Categorical Features:
['gender', 'stress_level', 'academic_work_impact']


In [14]:
# ==========================================================
# STEP 02 | HANDLE CATEGORICAL MISSING VALUES
# ==========================================================

X_train_cb = X_train.copy()
X_valid_cb = X_valid.copy()
X_test_cb = X_test.copy()

for column in categorical_features:
    X_train_cb[column] = X_train_cb[column].fillna("Missing")
    X_valid_cb[column] = X_valid_cb[column].fillna("Missing")
    X_test_cb[column] = X_test_cb[column].fillna("Missing")

print("Categorical missing values handled.")

Categorical missing values handled.


In [18]:
# ==========================================================
# STEP 03 | CATBOOST BASELINE
# ==========================================================

from catboost import CatBoostClassifier

catboost_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=7,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    allow_writing_files=False
)

catboost_model.fit(
    X_train_cb,
    y_train,
    cat_features=categorical_features,
    eval_set=(X_valid_cb, y_valid),
    use_best_model=True
)

0:	test: 0.9100706	best: 0.9100706 (0)	total: 586ms	remaining: 9m 45s
100:	test: 0.9389533	best: 0.9389533 (100)	total: 47.1s	remaining: 6m 59s
200:	test: 0.9447178	best: 0.9447178 (200)	total: 1m 30s	remaining: 6m 1s
300:	test: 0.9486249	best: 0.9486249 (300)	total: 2m 15s	remaining: 5m 14s
400:	test: 0.9517730	best: 0.9517730 (400)	total: 2m 59s	remaining: 4m 28s
500:	test: 0.9538514	best: 0.9538514 (500)	total: 3m 43s	remaining: 3m 42s
600:	test: 0.9553275	best: 0.9553275 (600)	total: 4m 27s	remaining: 2m 57s
700:	test: 0.9566871	best: 0.9566871 (700)	total: 5m 12s	remaining: 2m 13s
800:	test: 0.9576687	best: 0.9576687 (800)	total: 5m 56s	remaining: 1m 28s
900:	test: 0.9584021	best: 0.9584021 (900)	total: 6m 40s	remaining: 44s
999:	test: 0.9590402	best: 0.9590402 (999)	total: 7m 22s	remaining: 0us

bestTest = 0.9590402262
bestIteration = 999



CatBoostClassifier(allow_writing_files=False, depth=7, eval_metric='AUC', iterations=1000, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=100)

In [19]:
# ==========================================================
# STEP 04 | CATBOOST VALIDATION PREDICTIONS
# ==========================================================

y_valid_cb_proba = catboost_model.predict_proba(
    X_valid_cb
)[:, 1]

print("Validation predictions:", y_valid_cb_proba.shape)
print("Min probability:", y_valid_cb_proba.min())
print("Max probability:", y_valid_cb_proba.max())

Validation predictions: (138274,)
Min probability: 0.0005512744111884629
Max probability: 0.9999983984711215


In [20]:
# ==========================================================
# STEP 05 | CATBOOST ROC-AUC
# ==========================================================

from sklearn.metrics import roc_auc_score

catboost_auc = roc_auc_score(
    y_valid,
    y_valid_cb_proba
)

print(f"CatBoost ROC-AUC: {catboost_auc:.6f}")
print(f"Logistic ROC-AUC: {baseline_auc:.6f}")
print(f"Improvement     : {catboost_auc - baseline_auc:+.6f}")

CatBoost ROC-AUC: 0.959040
Logistic ROC-AUC: 0.917445
Improvement     : +0.041595


In [21]:
# ==========================================================
# STEP 06 | CATBOOST FEATURE IMPORTANCE
# ==========================================================

feature_importance_cb = (
    pd.DataFrame({
        "feature": X_train_cb.columns,
        "importance": catboost_model.get_feature_importance()
    })
    .sort_values("importance", ascending=False)
)

feature_importance_cb.head(15)

,feature,importance
8,weekend_screen_time,24.139255
1,daily_screen_time_hours,21.981835
6,notifications_per_day,12.128860
7,app_opens_per_day,10.912877
2,social_media_hours,10.338395
15,social_media_ratio,5.319683
17,work_study_ratio,3.411355
16,gaming_ratio,2.436313
4,work_study_hours,1.714277
19,screen_time_waking_ratio,1.254266


In [22]:
print("Top 15 CatBoost Features")
print(
    feature_importance_cb.head(15).to_string(index=False)
)

Top 15 CatBoost Features
                 feature  importance
     weekend_screen_time   24.139255
 daily_screen_time_hours   21.981835
   notifications_per_day   12.128860
       app_opens_per_day   10.912877
      social_media_hours   10.338395
      social_media_ratio    5.319683
        work_study_ratio    3.411355
            gaming_ratio    2.436313
        work_study_hours    1.714277
screen_time_waking_ratio    1.254266
         non_screen_time    1.251453
      weekend_difference    0.961167
     entertainment_ratio    0.944029
                     age    0.917947
            gaming_hours    0.752419


In [23]:
# ==========================================================
# EXPERIMENT LOG
# ==========================================================

experiment_results = pd.DataFrame([
    {
        "experiment": 1,
        "model": "Logistic Regression",
        "roc_auc": baseline_auc,
    },
    {
        "experiment": 2,
        "model": "CatBoost",
        "roc_auc": catboost_auc,
    },
])

experiment_results

,experiment,model,roc_auc
0,1,Logistic Regression,0.917445
1,2,CatBoost,0.959040


In [24]:
from src.modeling import train_catboost

In [25]:
# ==========================================================
# EXPERIMENT 03 | CATBOOST DEPTH 6
# ==========================================================

catboost_depth6 = train_catboost(
    X_train=X_train_cb,
    y_train=y_train,
    X_valid=X_valid_cb,
    y_valid=y_valid,
    categorical_features=categorical_features,
    depth=6,
    iterations=1000,
    learning_rate=0.05,
)

0:	test: 0.9040893	best: 0.9040893 (0)	total: 502ms	remaining: 8m 21s
100:	test: 0.9373743	best: 0.9373743 (100)	total: 48.4s	remaining: 7m 11s
200:	test: 0.9426199	best: 0.9426199 (200)	total: 1m 37s	remaining: 6m 28s
300:	test: 0.9469115	best: 0.9469115 (300)	total: 2m 23s	remaining: 5m 33s
400:	test: 0.9498995	best: 0.9498995 (400)	total: 3m 2s	remaining: 4m 31s
500:	test: 0.9520896	best: 0.9520896 (500)	total: 3m 39s	remaining: 3m 38s
600:	test: 0.9536992	best: 0.9536992 (600)	total: 4m 17s	remaining: 2m 51s
700:	test: 0.9548647	best: 0.9548647 (700)	total: 4m 56s	remaining: 2m 6s
800:	test: 0.9560951	best: 0.9560951 (800)	total: 5m 34s	remaining: 1m 23s
900:	test: 0.9569878	best: 0.9569878 (900)	total: 6m 13s	remaining: 41s
999:	test: 0.9577009	best: 0.9577009 (999)	total: 6m 50s	remaining: 0us

bestTest = 0.9577009391
bestIteration = 999



In [27]:
y_valid_depth6 = catboost_depth6.predict_proba(
    X_valid_cb
)[:, 1]

auc_depth6 = roc_auc_score(
    y_valid,
    y_valid_depth6
)

print(f"CatBoost depth=6 ROC-AUC: {auc_depth6:.6f}")
print(f"Current best ROC-AUC     : {catboost_auc:.6f}")
print(f"Improvement              : {auc_depth6 - catboost_auc:+.6f}")

CatBoost depth=6 ROC-AUC: 0.957701
Current best ROC-AUC     : 0.959040
Improvement              : -0.001339


In [28]:
# ==========================================================
# EXPERIMENT 04 | CATBOOST DEPTH 8
# ==========================================================

catboost_depth8 = train_catboost(
    X_train=X_train_cb,
    y_train=y_train,
    X_valid=X_valid_cb,
    y_valid=y_valid,
    categorical_features=categorical_features,
    depth=8,
    iterations=1000,
    learning_rate=0.05,
)

y_valid_depth8 = catboost_depth8.predict_proba(
    X_valid_cb
)[:, 1]

auc_depth8 = roc_auc_score(
    y_valid,
    y_valid_depth8
)

print(f"CatBoost depth=8 ROC-AUC: {auc_depth8:.6f}")
print(f"Current best ROC-AUC     : {catboost_auc:.6f}")
print(f"Improvement              : {auc_depth8 - catboost_auc:+.6f}")

0:	test: 0.9106364	best: 0.9106364 (0)	total: 705ms	remaining: 11m 44s
100:	test: 0.9401146	best: 0.9401146 (100)	total: 1m 3s	remaining: 9m 28s
200:	test: 0.9461667	best: 0.9461667 (200)	total: 2m 6s	remaining: 8m 22s
300:	test: 0.9500487	best: 0.9500487 (300)	total: 3m 12s	remaining: 7m 26s
400:	test: 0.9530771	best: 0.9530771 (400)	total: 4m 17s	remaining: 6m 23s
500:	test: 0.9551284	best: 0.9551284 (500)	total: 5m 12s	remaining: 5m 11s
600:	test: 0.9565812	best: 0.9565812 (600)	total: 6m 8s	remaining: 4m 4s
700:	test: 0.9576445	best: 0.9576445 (700)	total: 7m 25s	remaining: 3m 10s
800:	test: 0.9585481	best: 0.9585481 (800)	total: 8m 24s	remaining: 2m 5s
900:	test: 0.9591718	best: 0.9591718 (900)	total: 9m 29s	remaining: 1m 2s
999:	test: 0.9597599	best: 0.9597599 (999)	total: 10m 38s	remaining: 0us

bestTest = 0.9597599364
bestIteration = 999

CatBoost depth=8 ROC-AUC: 0.959760
Current best ROC-AUC     : 0.959040
Improvement              : +0.000720


In [29]:
 # ==========================================================
# EXPERIMENT 05 | CATBOOST DEPTH 9
# ==========================================================

catboost_depth9 = train_catboost(
    X_train=X_train_cb,
    y_train=y_train,
    X_valid=X_valid_cb,
    y_valid=y_valid,
    categorical_features=categorical_features,
    depth=9,
    iterations=1000,
    learning_rate=0.05,
)

y_valid_depth9 = catboost_depth9.predict_proba(
    X_valid_cb
)[:, 1]

auc_depth9 = roc_auc_score(
    y_valid,
    y_valid_depth9
)

print(f"CatBoost depth=9 ROC-AUC: {auc_depth9:.6f}")
print(f"Current best ROC-AUC     : {catboost_auc:.6f}")
print(f"Improvement              : {auc_depth9 - catboost_auc:+.6f}")

0:	test: 0.9124075	best: 0.9124075 (0)	total: 790ms	remaining: 13m 9s
100:	test: 0.9412554	best: 0.9412554 (100)	total: 1m 27s	remaining: 13m 1s
200:	test: 0.9471920	best: 0.9471920 (200)	total: 2m 38s	remaining: 10m 29s
300:	test: 0.9512200	best: 0.9512200 (300)	total: 3m 48s	remaining: 8m 49s
400:	test: 0.9540335	best: 0.9540335 (400)	total: 4m 58s	remaining: 7m 26s
500:	test: 0.9561096	best: 0.9561096 (500)	total: 6m 9s	remaining: 6m 8s
600:	test: 0.9574317	best: 0.9574317 (600)	total: 7m 22s	remaining: 4m 53s
700:	test: 0.9584982	best: 0.9584982 (700)	total: 8m 36s	remaining: 3m 40s
800:	test: 0.9591776	best: 0.9591776 (800)	total: 9m 42s	remaining: 2m 24s
900:	test: 0.9598744	best: 0.9598744 (900)	total: 10m 49s	remaining: 1m 11s
999:	test: 0.9603846	best: 0.9603846 (999)	total: 11m 57s	remaining: 0us

bestTest = 0.9603846116
bestIteration = 999

CatBoost depth=9 ROC-AUC: 0.960385
Current best ROC-AUC     : 0.959040
Improvement              : +0.001344


In [30]:
best_auc = max(
    baseline_auc,
    catboost_auc,
    auc_depth6,
    auc_depth8
)

print(f"Best ROC-AUC: {best_auc:.6f}")
print(f"Depth 9 improvement: {auc_depth9 - best_auc:+.6f}")

Best ROC-AUC: 0.959760
Depth 9 improvement: +0.000625


In [32]:
# ==========================================================
# EXPERIMENT 06 | CATBOOST DEPTH 10
# ==========================================================

catboost_depth10 = train_catboost(
    X_train=X_train_cb,
    y_train=y_train,
    X_valid=X_valid_cb,
    y_valid=y_valid,
    categorical_features=categorical_features,
    depth=10,
    iterations=2000,
    learning_rate=0.05,
)

y_valid_depth10 = catboost_depth10.predict_proba(
    X_valid_cb
)[:, 1]

auc_depth10 = roc_auc_score(
    y_valid,
    y_valid_depth10
)

best_auc = max(
    baseline_auc,
    catboost_auc,
    auc_depth6,
    auc_depth8,
    auc_depth9
)

print(f"CatBoost depth=10 ROC-AUC: {auc_depth10:.6f}")
print(f"Current best ROC-AUC      : {best_auc:.6f}")
print(f"Improvement               : {auc_depth10 - best_auc:+.6f}")

0:	test: 0.9149785	best: 0.9149785 (0)	total: 1.13s	remaining: 37m 43s
100:	test: 0.9424866	best: 0.9424866 (100)	total: 1m 29s	remaining: 27m 58s
200:	test: 0.9484412	best: 0.9484412 (200)	total: 2m 48s	remaining: 25m 4s
300:	test: 0.9524510	best: 0.9524510 (300)	total: 4m 20s	remaining: 24m 31s
400:	test: 0.9552254	best: 0.9552254 (400)	total: 5m 53s	remaining: 23m 28s
500:	test: 0.9570006	best: 0.9570006 (500)	total: 7m 13s	remaining: 21m 37s
600:	test: 0.9582475	best: 0.9582475 (600)	total: 8m 31s	remaining: 19m 49s
700:	test: 0.9592303	best: 0.9592303 (699)	total: 9m 50s	remaining: 18m 14s
800:	test: 0.9597732	best: 0.9597732 (800)	total: 11m 6s	remaining: 16m 38s
900:	test: 0.9603069	best: 0.9603069 (900)	total: 12m 24s	remaining: 15m 8s
1000:	test: 0.9608057	best: 0.9608057 (1000)	total: 13m 41s	remaining: 13m 40s
1100:	test: 0.9611168	best: 0.9611174 (1099)	total: 15m 29s	remaining: 12m 39s
1200:	test: 0.9613503	best: 0.9613503 (1200)	total: 17m 2s	remaining: 11m 20s
1300:	test

In [33]:
# ==========================================================
# FINAL MODEL | STEP 01
# PREPARE FULL TRAINING + TEST DATA
# ==========================================================

train_final = train_fe.copy()
test_final = test_fe.copy()

y_final = train_final["addicted_label"]

X_final = train_final.drop(
    columns=["id", "addicted_label"]
)

X_test_final = test_final.drop(
    columns=["id"]
)

categorical_features = X_final.select_dtypes(
    include=["object"]
).columns.tolist()

for column in categorical_features:
    X_final[column] = X_final[column].fillna("Missing")
    X_test_final[column] = X_test_final[column].fillna("Missing")

print("X_final      :", X_final.shape)
print("y_final      :", y_final.shape)
print("X_test_final :", X_test_final.shape)
print("Categorical  :", categorical_features)

X_final      : (691369, 21)
y_final      : (691369,)
X_test_final : (296302, 21)
Categorical  : ['gender', 'stress_level', 'academic_work_impact']


In [34]:
# ==========================================================
# FINAL MODEL | STEP 02
# TRAIN ON ALL LABELED DATA
# ==========================================================

from catboost import CatBoostClassifier

final_model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.05,
    depth=10,
    loss_function="Logloss",
    random_seed=42,
    verbose=100,
    allow_writing_files=False
)

final_model.fit(
    X_final,
    y_final,
    cat_features=categorical_features
)

print("Final model training completed.")

0:	learn: 0.6327291	total: 1.01s	remaining: 33m 49s
100:	learn: 0.2719489	total: 1m 53s	remaining: 35m 26s
200:	learn: 0.2575552	total: 3m 36s	remaining: 32m 17s
300:	learn: 0.2477946	total: 5m 14s	remaining: 29m 32s
400:	learn: 0.2399788	total: 7m 14s	remaining: 28m 53s
500:	learn: 0.2348567	total: 8m 59s	remaining: 26m 55s
600:	learn: 0.2304227	total: 10m 34s	remaining: 24m 36s
700:	learn: 0.2266969	total: 12m 6s	remaining: 22m 26s
800:	learn: 0.2234193	total: 13m 32s	remaining: 20m 16s
900:	learn: 0.2207007	total: 15m 4s	remaining: 18m 23s
1000:	learn: 0.2181020	total: 16m 48s	remaining: 16m 46s
1100:	learn: 0.2155358	total: 18m 23s	remaining: 15m
1200:	learn: 0.2130198	total: 20m 3s	remaining: 13m 20s
1300:	learn: 0.2105419	total: 21m 43s	remaining: 11m 40s
1400:	learn: 0.2081781	total: 23m 22s	remaining: 9m 59s
1500:	learn: 0.2059669	total: 25m 21s	remaining: 8m 25s
1600:	learn: 0.2037626	total: 27m 17s	remaining: 6m 48s
1700:	learn: 0.2014555	total: 29m 9s	remaining: 5m 7s
1800:	

In [35]:
# ==========================================================
# FINAL MODEL | STEP 03
# TEST PREDICTIONS
# ==========================================================

test_proba = final_model.predict_proba(
    X_test_final
)[:, 1]  

print("Prediction shape:", test_proba.shape)
print("Minimum:", test_proba.min())
print("Maximum:", test_proba.max())

Prediction shape: (296302,)
Minimum: 0.0001315475742815282
Maximum: 0.9999999088152761


In [36]:
# ==========================================================
# FINAL MODEL | STEP 04
# CREATE SUBMISSION
# ==========================================================

import pandas as pd

submission = pd.DataFrame({
    "id": test_final["id"],
    "addicted_label": test_proba
})

print(submission.shape)
print(submission.head())

(296302, 2)
       id  addicted_label
0  691369        0.999388
1  691370        0.931489
2  691371        0.964222
3  691372        0.991131
4  691373        0.995558


In [37]:
# ==========================================================
# FINAL MODEL | STEP 05
# SUBMISSION VALIDATION
# ==========================================================

print("Rows:", len(submission))

print(
    "Duplicate IDs:",
    submission["id"].duplicated().sum()
)

print(
    "Missing IDs:",
    submission["id"].isna().sum()
)

print(
    "Missing predictions:",
    submission["addicted_label"].isna().sum()
)

print(
    "Prediction min:",
    submission["addicted_label"].min()
)

print(
    "Prediction max:",
    submission["addicted_label"].max()
)

print(
    "IDs match test:",
    submission["id"].equals(test_final["id"])
)

Rows: 296302
Duplicate IDs: 0
Missing IDs: 0
Missing predictions: 0
Prediction min: 0.0001315475742815282
Prediction max: 0.9999999088152761
IDs match test: True


In [39]:
# ==========================================================
# CREATE SUBMISSION DIRECTORY
# ==========================================================

import os

os.makedirs("../submissions", exist_ok=True)

submission_path = "../submissions/submission_catboost_d10_2000.csv"

submission.to_csv(
    submission_path,
    index=False
)

print("Submission saved successfully!")
print("Path:", submission_path)

Submission saved successfully!
Path: ../submissions/submission_catboost_d10_2000.csv


In [40]:
import os

print("File exists:", os.path.exists(submission_path))

if os.path.exists(submission_path):
    print("File size:", os.path.getsize(submission_path), "bytes")

File exists: True
File size: 8055205 bytes
